# DASH-NN Phase 0: Go/No-Go Validation

Minimal experiment to test whether averaging NN attributions via DASH improves stability.

**Config:** M=50, K=20, N_REPS=5, rho=0.9, P=50, N=5000

**Go criterion:** DASH-NN stability > SingleNN stability by >= 0.02 at rho=0.9

In [ ]:
import numpy as np
import time

from dash_shap.core.pipeline import DASHPipeline
from dash_shap.core.nn_population import generate_nn_population
from dash_shap.core.nn_attribution import compute_nn_attributions
from dash_shap.core.filtering import performance_filter
from dash_shap.baselines.nn_baselines import SingleNNBaseline, BaggedNNBaseline
from dash_shap.experiments.synthetic import generate_synthetic_linear
from dash_shap.evaluation import stability_bootstrap_ci

PHASE0_CONFIG = {
    "M": 50,
    "K": 20,
    "N_REPS": 5,
    "EPSILON": 0.08,
    "SEED": 42,
    "N": 5000,
    "P": 50,
    "RHO": 0.9,
    "GROUP_SIZE": 5,
}
print("Phase 0 config:", PHASE0_CONFIG)

## 1. Generate Synthetic Data

In [ ]:
cfg = PHASE0_CONFIG
Xtr, ytr, Xv, yv, Xexp, yexp, Xte, yte, groups, true_imp, meta = generate_synthetic_linear(
    N=cfg["N"], P=cfg["P"], group_size=cfg["GROUP_SIZE"], rho=cfg["RHO"], seed=cfg["SEED"]
)
print(f"Train: {Xtr.shape}, Val: {Xv.shape}, Explain: {Xexp.shape}, Test: {Xte.shape}")
print(f"True importance (top 10): {np.argsort(true_imp)[-10:][::-1].tolist()}")

## 2. Train NN Population

In [ ]:
t0 = time.time()
models, val_scores, configs = generate_nn_population(
    Xtr, ytr, Xv, yv, M=cfg["M"], seed=cfg["SEED"], n_jobs=-1
)
print(f"Population trained in {time.time() - t0:.1f}s")
scores = list(val_scores.values())
print(f"Val score distribution: mean={np.mean(scores):.4f}, std={np.std(scores):.4f}")

## 3. Compute KernelSHAP Attributions

In [ ]:
# Use a small reference set for speed
X_ref = Xexp[:100]
all_indices = list(range(cfg["M"]))

t0 = time.time()
consensus, all_shap = compute_nn_attributions(
    models, all_indices, X_ref, seed=cfg["SEED"], n_jobs=-1, verbose=True
)
elapsed = time.time() - t0
print(f"KernelSHAP computed in {elapsed:.1f}s ({elapsed / cfg['M']:.1f}s/model)")

## 4. Run DASH-NN via fit_from_attributions()

In [ ]:
pipe = DASHPipeline(
    M=cfg["M"], K=cfg["K"], epsilon=cfg["EPSILON"], seed=cfg["SEED"]
)
pipe.fit_from_attributions(all_shap, val_scores)

print(f"Selected {len(pipe.selected_indices_)} models")
print(f"DASH-NN top 10 features: {np.argsort(pipe.global_importance_)[-10:][::-1].tolist()}")
print(f"FSI (mean): {np.mean(pipe.fsi_):.4f}")

## 5. Baselines

In [ ]:
# SingleNN baseline
single_nn = SingleNNBaseline(n_trials=cfg["M"], seed=cfg["SEED"])
single_nn.fit(Xtr, ytr, Xv, yv, X_ref=X_ref)
print(f"SingleNN top 10: {np.argsort(single_nn.global_importance_)[-10:][::-1].tolist()}")

# BaggedNN baseline
bagged_nn = BaggedNNBaseline(N=cfg["K"], seed=cfg["SEED"], n_jobs=-1)
bagged_nn.fit(Xtr, ytr, Xv, yv, X_ref=X_ref)
print(f"BaggedNN top 10: {np.argsort(bagged_nn.global_importance_)[-10:][::-1].tolist()}")

## 6. Stability Comparison (N_REPS repetitions)

In [ ]:
results = {"DASH-NN": [], "SingleNN": [], "BaggedNN": []}

for rep in range(cfg["N_REPS"]):
    rep_seed = cfg["SEED"] + rep * 1000
    print(f"\n--- Rep {rep + 1}/{cfg['N_REPS']} (seed={rep_seed}) ---")

    # Generate fresh data
    Xtr_r, ytr_r, Xv_r, yv_r, Xexp_r, yexp_r, *_ = generate_synthetic_linear(
        N=cfg["N"], P=cfg["P"], group_size=cfg["GROUP_SIZE"],
        rho=cfg["RHO"], seed=rep_seed
    )
    X_ref_r = Xexp_r[:100]

    # DASH-NN
    models_r, scores_r, _ = generate_nn_population(
        Xtr_r, ytr_r, Xv_r, yv_r, M=cfg["M"], seed=rep_seed, n_jobs=-1, verbose=False
    )
    _, all_shap_r = compute_nn_attributions(
        models_r, list(range(cfg["M"])), X_ref_r,
        seed=rep_seed, n_jobs=-1, verbose=False
    )
    pipe_r = DASHPipeline(M=cfg["M"], K=cfg["K"], epsilon=cfg["EPSILON"], seed=rep_seed)
    pipe_r.fit_from_attributions(all_shap_r, scores_r)
    results["DASH-NN"].append(pipe_r.global_importance_)

    # SingleNN
    snn = SingleNNBaseline(n_trials=cfg["M"], seed=rep_seed)
    snn.fit(Xtr_r, ytr_r, Xv_r, yv_r, X_ref=X_ref_r)
    results["SingleNN"].append(snn.global_importance_)

    # BaggedNN
    bnn = BaggedNNBaseline(N=cfg["K"], seed=rep_seed, n_jobs=-1)
    bnn.fit(Xtr_r, ytr_r, Xv_r, yv_r, X_ref=X_ref_r)
    results["BaggedNN"].append(bnn.global_importance_)

print("\nAll reps complete.")

In [ ]:
print("=" * 60)
print("STABILITY RESULTS")
print("=" * 60)

for method, importances in results.items():
    imp_matrix = np.array(importances)  # (N_REPS, P)
    stat, ci_lo, ci_hi = stability_bootstrap_ci(imp_matrix)
    print(f"{method:12s}: stability={stat:.4f}  95% CI=[{ci_lo:.4f}, {ci_hi:.4f}]")

# Go/no-go
dash_stab = stability_bootstrap_ci(np.array(results["DASH-NN"]))[0]
single_stab = stability_bootstrap_ci(np.array(results["SingleNN"]))[0]
delta = dash_stab - single_stab
print(f"\nDelta (DASH-NN - SingleNN): {delta:.4f}")
print(f"GO/NO-GO: {'PASS' if delta >= 0.02 else 'FAIL'} (threshold=0.02)")

## 7. DGP Agreement

In [ ]:
from scipy.stats import spearmanr

print("DGP Agreement (Spearman correlation with true importance):")
for method, importances in results.items():
    avg_imp = np.mean(importances, axis=0)
    corr, _ = spearmanr(avg_imp, true_imp)
    print(f"  {method:12s}: rho={corr:.4f}")